<a href="https://colab.research.google.com/github/fboldt/aulas-am-bsi/blob/main/aula05a_valida%C3%A7%C3%A3o_cruzada.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

X, y = load_wine(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = KNeighborsClassifier()
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.9444444444444444


In [5]:
model = KNeighborsClassifier(n_neighbors=15)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.9722222222222222


In [22]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

k = 5
model = KNeighborsClassifier(n_neighbors=k)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy n_neighbors={k:>2}:", accuracy)

k = 15
model = KNeighborsClassifier(n_neighbors=k)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy n_neighbors={k:>2}:", accuracy)

Accuracy n_neighbors= 5: 0.9444444444444444
Accuracy n_neighbors=15: 0.9166666666666666


In [46]:
from sklearn.pipeline import Pipeline
from pprint import pprint
import numpy as np


models = [KNeighborsClassifier(3), KNeighborsClassifier(15)]

def evalute_model(model, X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    scaler = StandardScaler()
    scaler.fit(X_train)
    X_train_scaled = scaler.transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    accuracy = accuracy_score(y_test, y_pred)
    return accuracy

def multiple_holdout(models, X, y, n_repeats=5):
    accuracies = {}
    for model in models:
        accuracies[model] = []
    for i in range(n_repeats):
        for model in models:
            accuracy = evalute_model(model, X, y)
            accuracies[model].append(accuracy)
    return accuracies

accuracies = multiple_holdout(models, X, y)
pprint(accuracies)

{KNeighborsClassifier(n_neighbors=15): [0.9722222222222222,
                                        0.8888888888888888,
                                        0.9166666666666666,
                                        0.9444444444444444,
                                        1.0],
 KNeighborsClassifier(n_neighbors=3): [1.0,
                                       0.9166666666666666,
                                       0.9722222222222222,
                                       0.9444444444444444,
                                       0.9722222222222222]}


In [50]:
def cross_validation(models, X, y, n_folds=5):
    accuracies = {}
    for model in models:
        accuracies[model] = []
    idxs = np.random.permutation(len(X))
    test_size = len(X) // n_folds
    for i in range(n_folds):
        test_idxs = idxs[i*test_size:(i+1)*test_size]
        train_idxs = np.concatenate([idxs[:i*test_size], idxs[(i+1)*test_size:]])
        X_train, y_train = X[train_idxs], y[train_idxs]
        X_test, y_test = X[test_idxs], y[test_idxs]
        scaler = StandardScaler()
        scaler.fit(X_train)
        X_train_scaled = scaler.transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        for model in models:
            model.fit(X_train_scaled, y_train)
            y_pred = model.predict(X_test_scaled)
            accuracy = accuracy_score(y_test, y_pred)
            accuracies[model].append(accuracy)
    return accuracies

accuracies = cross_validation(models, X, y)
pprint(accuracies)


{KNeighborsClassifier(n_neighbors=15): [0.9714285714285714,
                                        0.9714285714285714,
                                        1.0,
                                        0.9428571428571428,
                                        1.0],
 KNeighborsClassifier(n_neighbors=3): [0.9428571428571428,
                                       1.0,
                                       0.9428571428571428,
                                       0.9714285714285714,
                                       0.9714285714285714]}


In [61]:
from sklearn.model_selection import cross_validate

models = [
    Pipeline([("scaler", StandardScaler()), ("model", KNeighborsClassifier(3))]),
    Pipeline([("scaler", StandardScaler()), ("model", KNeighborsClassifier(15))])
]

for model in models:
  scores = cross_validate(model, X, y, cv=5, return_train_score=True)
  pprint(scores["test_score"])

array([0.88888889, 0.94444444, 0.97222222, 1.        , 0.91428571])
array([0.91666667, 0.94444444, 0.97222222, 1.        , 0.94285714])


In [68]:
from sklearn.model_selection import StratifiedKFold

splitter = StratifiedKFold(n_splits=5)

for model in models:
  scores = cross_validate(model, X, y, cv=splitter, return_train_score=True)
  pprint(scores["test_score"])

array([0.88888889, 0.94444444, 0.97222222, 1.        , 0.91428571])
array([0.91666667, 0.94444444, 0.97222222, 1.        , 0.94285714])


In [72]:
from sklearn.model_selection import KFold

splitter = KFold(n_splits=5)

for model in models:
  scores = cross_validate(model, X, y, cv=splitter, return_train_score=True)
  pprint(scores["test_score"])

array([0.94444444, 0.91666667, 0.80555556, 0.88571429, 0.97142857])
array([0.97222222, 0.94444444, 0.80555556, 0.94285714, 0.97142857])


In [79]:
from sklearn.model_selection import KFold

splitter = KFold(n_splits=5, shuffle=True)

for model in models:
  scores = cross_validate(model, X, y, cv=splitter, return_train_score=True)
  pprint(scores["test_score"])

array([0.94444444, 0.94444444, 0.88888889, 0.97142857, 1.        ])
array([0.91666667, 0.94444444, 0.91666667, 0.97142857, 0.94285714])


In [81]:
from sklearn.model_selection import LeaveOneOut

splitter = LeaveOneOut()

for model in models:
  scores = cross_validate(model, X, y, cv=splitter, return_train_score=True)
  pprint(scores["test_score"].mean())

np.float64(0.9550561797752809)
np.float64(0.9662921348314607)


In [83]:
from sklearn.model_selection import RepeatedKFold

splitter = RepeatedKFold(n_splits=5, n_repeats=10)

for model in models:
  scores = cross_validate(model, X, y, cv=splitter, return_train_score=True)
  pprint(scores["test_score"].mean())
#

np.float64(0.9544603174603173)
np.float64(0.9669047619047617)
